In [ ]:
import json
from openai import OpenAI

# 1. Connect to Ollama
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

MODEL_ID = "qwen3:4b"

# 2. User query
question = "How many annual leave days do employees receive?"

# 3. Retrieved contexts
retrieved_contexts = [
    "Employees receive 26 days of annual leave per year.",
    "Employees are covered by supplementary health insurance.",
    "Employees can work remotely up to 2 days per week.",
]

# 4. Evaluation prompt
prompt = f"""
You are an expert RAG evaluation judge.

Evaluate how relevant each retrieved context is
to answering the user's question.

User Question:
{question}

Retrieved Contexts:
{json.dumps(retrieved_contexts, indent=2)}

Use this scoring rubric:

0 = Completely irrelevant.
1 = Partially relevant.
2 = Fully relevant and useful.

Evaluate each context separately.

Return ONLY valid JSON in this format:

{{
    "evaluations": [
        {{
            "context_index": 0,
            "score": 2,
            "reason": "Explanation"
        }}
    ]
}}

Do not include Markdown or additional text.
"""

# 5. Ask Qwen3 to evaluate
response = client.chat.completions.create(
    model=MODEL_ID, messages=[{"role": "user", "content": prompt}], temperature=0
)

# 6. Parse the model response
result_text = response.choices[0].message.content

try:
    result = json.loads(result_text)

    evaluations = result["evaluations"]

    scores = []

    for item in evaluations:
        index = item["context_index"]
        score = item["score"]

        if not isinstance(index, int) or not 0 <= index < len(retrieved_contexts):
            raise ValueError("Invalid context index.")

        if not isinstance(score, int) or score not in (0, 1, 2):
            raise ValueError("Invalid relevance score.")

        scores.append(score)

    if len(scores) != len(retrieved_contexts):
        raise ValueError("The judge did not evaluate every context.")

    # Normalize average score to [0, 1]
    context_relevance = sum(scores) / (2 * len(scores))

    print("Context Evaluations:")

    for item in evaluations:
        print(f"\nContext {item['context_index']}:")
        print("Score:", item["score"])
        print("Reason:", item["reason"])

    print(f"\nContext Relevance: {context_relevance:.2f}")

except (json.JSONDecodeError, KeyError, TypeError, ValueError) as error:
    print("Could not parse a valid evaluation.")
    print("Error:", error)
    print("Model response:", result_text)